# Singapore IT Jobs: Data Cleaning 

**Question:** How do advertised monthly salary ranges differ across IT position levels?

One row represents one job posting. This notebook prepares the team dataset using the relevant ideas from [Lesson 1.8: EDA Basic](https://github.com/85913286lcr/6m-data-1.8-eda-basic/blob/main/lesson.md).

1. Descriptive statistics
2. Data quality
3. Data transformation and a simple summary
4. Export and check

**Note:** Each decision follows **find → decide → apply → verify**. Salary figures describe advertised ranges, not actual employee pay.

## Setup

**Note:** Load the complete source file and keep an unchanged copy for comparison.

In [13]:
import pandas as pd
import json
from pathlib import Path

source_candidates = [Path("SGJobData.csv"), Path("../SGJobData.csv")]
source = next((path for path in source_candidates if path.exists()), None)
if source is None:
    raise FileNotFoundError("SGJobData.csv was not found in the current or parent directory.")

raw = pd.read_csv(source, low_memory=False)
clean = raw.copy()
print("Source:", source)
print("Raw rows:", len(raw))


Source: ../SGJobData.csv
Raw rows: 1048585


## Part 1: Descriptive Statistics
### 1.1 First look

**Note:** Use `head`, `shape`, `info`, `dtypes` and `describe` to understand the rows, column types and numerical ranges before changing anything.

In [14]:
display(raw.head())
print("Rows and columns:", raw.shape)
raw.info()
display(raw.dtypes)
display(raw.describe())

,categories,employmentTypes,metadata_expiryDate,metadata_isPostedOnBehalf,metadata_jobPostId,metadata_newPostingDate,metadata_originalPostingDate,metadata_repostCount,metadata_totalNumberJobApplication,metadata_totalNumberOfView,...,occupationId,positionLevels,postedCompany_name,salary_maximum,salary_minimum,salary_type,status_id,status_jobStatus,title,average_salary
0,"[{""id"":13,""category"":""Environment / Health""},{...",Permanent,2023-05-08,False,MCF-2023-0252866,2023-04-08,2023-03-30,2,5,151,...,NaN,Executive,WORKSTONE PTE. LTD.,2800,2000,Monthly,0,Closed,Food Technologist - Clementi | Entry Level | U...,2400.0
1,"[{""id"":21,""category"":""Information Technology""}]",Permanent,2023-05-08,False,MCF-2023-0273977,2023-04-08,2023-04-08,0,0,55,...,NaN,Executive,TRUST RECRUIT PTE. LTD.,5500,4000,Monthly,0,Closed,"Software Engineer (Fab Support) (Java, CIM, Up...",4750.0
2,"[{""id"":33,""category"":""Repair and Maintenance""}]",Full Time,2023-04-22,False,MCF-2023-0273994,2023-04-08,2023-04-08,0,7,99,...,NaN,Senior Executive,PU TIEN SERVICES PTE. LTD.,4600,3800,Monthly,0,Closed,Senior Technician,4200.0
3,"[{""id"":21,""category"":""Information Technology""}]",Permanent,2023-05-08,False,MCF-2023-0273991,2023-04-08,2023-04-08,0,6,113,...,NaN,Senior Executive,TRUST RECRUIT PTE. LTD.,10000,5000,Monthly,0,Closed,"Senior .NET Developer (.NET Core, MVC, MVVC, S...",7500.0
4,"[{""id"":2,""category"":""Admin / Secretarial""}]",Full Time,2023-05-08,False,MCF-2023-0273976,2023-04-08,2023-04-08,0,3,99,...,NaN,Non-executive,EATZ CATERING SERVICES PTE. LTD.,3400,2400,Monthly,0,Closed,Sales / Admin Cordinator,2900.0


Rows and columns: (1048585, 22)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048585 entries, 0 to 1048584
Data columns (total 22 columns):
 #   Column                              Non-Null Count    Dtype  
---  ------                              --------------    -----  
 0   categories                          1044597 non-null  object 
 1   employmentTypes                     1044597 non-null  object 
 2   metadata_expiryDate                 1044597 non-null  object 
 3   metadata_isPostedOnBehalf           1048585 non-null  bool   
 4   metadata_jobPostId                  1044597 non-null  object 
 5   metadata_newPostingDate             1044597 non-null  object 
 6   metadata_originalPostingDate        1044597 non-null  object 
 7   metadata_repostCount                1048585 non-null  int64  
 8   metadata_totalNumberJobApplication  1048585 non-null  int64  
 9   metadata_totalNumberOfView          1048585 non-null  int64  
 10  minimumYearsExperience              1048585 no

categories                             object
employmentTypes                        object
metadata_expiryDate                    object
metadata_isPostedOnBehalf                bool
metadata_jobPostId                     object
metadata_newPostingDate                object
metadata_originalPostingDate           object
metadata_repostCount                    int64
metadata_totalNumberJobApplication      int64
metadata_totalNumberOfView              int64
minimumYearsExperience                  int64
numberOfVacancies                       int64
occupationId                          float64
positionLevels                         object
postedCompany_name                     object
salary_maximum                          int64
salary_minimum                          int64
salary_type                            object
status_id                               int64
status_jobStatus                       object
title                                  object
average_salary                    

,metadata_repostCount,metadata_totalNumberJobApplication,metadata_totalNumberOfView,minimumYearsExperience,numberOfVacancies,occupationId,salary_maximum,salary_minimum,status_id,average_salary
count,1.048585e+06,1.048585e+06,1.048585e+06,1.048585e+06,1.048585e+06,0.0,1.048585e+06,1.048585e+06,1048585.0,1.048585e+06
mean,5.472327e-02,2.136571e+00,2.674536e+01,2.779573e+00,2.680043e+00,NaN,5.723578e+03,3.815312e+03,0.0,4.769445e+03
std,2.822675e-01,1.062612e+01,8.262001e+01,2.537049e+00,1.124301e+01,NaN,5.018387e+04,3.172182e+03,0.0,2.547809e+04
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,NaN,0.000000e+00,0.000000e+00,0.0,0.000000e+00
25%,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,NaN,3.300000e+03,2.500000e+03,0.0,2.900000e+03
50%,0.000000e+00,0.000000e+00,4.000000e+00,2.000000e+00,1.000000e+00,NaN,4.500000e+03,3.000000e+03,0.0,3.800000e+03
75%,0.000000e+00,1.000000e+00,1.700000e+01,4.000000e+00,2.000000e+00,NaN,6.500000e+03,4.500000e+03,0.0,5.500000e+03
max,2.000000e+00,1.342000e+03,8.190000e+03,8.800000e+01,9.990000e+02,NaN,2.533000e+07,3.500000e+05,0.0,1.266640e+07


### 1.2 Categories and missing values

**Note:** Frequency counts show the labels used in the source. Missing values are checked by column; a missing value does not automatically mean the whole row should be removed.

In [15]:
display(raw["positionLevels"].value_counts(dropna=False))
display(raw["salary_type"].value_counts(dropna=False))
missing_summary = pd.DataFrame({
    "missing_count": raw.isna().sum(),
    "missing_percent": (raw.isna().mean() * 100).round(2)
})
display(missing_summary.sort_values("missing_count", ascending=False))

Executive            253701
Junior Executive     167656
Non-executive        131608
Fresh/entry level    118661
Professional         112208
Manager              110122
Senior Executive     100459
Middle Management     27375
Senior Management     22807
NaN                    3988
Name: positionLevels, dtype: int64

Monthly    1044597
NaN           3988
Name: salary_type, dtype: int64

,missing_count,missing_percent
occupationId,1048585,100.00
categories,3988,0.38
metadata_expiryDate,3988,0.38
title,3988,0.38
metadata_jobPostId,3988,0.38
metadata_newPostingDate,3988,0.38
metadata_originalPostingDate,3988,0.38
status_jobStatus,3988,0.38
salary_type,3988,0.38
employmentTypes,3988,0.38


## Part 2: Data Quality
### 2.1 Structurally empty records and an empty column

**Note:** Remove a row only when title, categories and posting date are all missing, and salary bounds and vacancies are zero or missing. Remove `occupationId` only if every value is missing. Other missing fields are retained without imputation.

In [16]:
structurally_empty = (
    clean["title"].isna()
    & clean["categories"].isna()
    & clean["metadata_originalPostingDate"].isna()
    & clean["salary_minimum"].fillna(0).eq(0)
    & clean["salary_maximum"].fillna(0).eq(0)
    & clean["numberOfVacancies"].fillna(0).eq(0)
)
display(clean.loc[structurally_empty].head())
clean = clean.loc[~structurally_empty].copy()
print("Structurally empty rows removed:", structurally_empty.sum())

if clean["occupationId"].isna().all():
    clean = clean.drop(columns="occupationId")
    print("Removed the entirely missing occupationId column.")
print("Remaining rows:", len(clean))

,categories,employmentTypes,metadata_expiryDate,metadata_isPostedOnBehalf,metadata_jobPostId,metadata_newPostingDate,metadata_originalPostingDate,metadata_repostCount,metadata_totalNumberJobApplication,metadata_totalNumberOfView,...,occupationId,positionLevels,postedCompany_name,salary_maximum,salary_minimum,salary_type,status_id,status_jobStatus,title,average_salary
197478,NaN,NaN,NaN,False,NaN,NaN,NaN,0,0,0,...,NaN,NaN,NaN,0,0,NaN,0,NaN,NaN,0.0
197480,NaN,NaN,NaN,False,NaN,NaN,NaN,0,0,0,...,NaN,NaN,NaN,0,0,NaN,0,NaN,NaN,0.0
197485,NaN,NaN,NaN,False,NaN,NaN,NaN,0,0,0,...,NaN,NaN,NaN,0,0,NaN,0,NaN,NaN,0.0
197488,NaN,NaN,NaN,False,NaN,NaN,NaN,0,0,0,...,NaN,NaN,NaN,0,0,NaN,0,NaN,NaN,0.0
197502,NaN,NaN,NaN,False,NaN,NaN,NaN,0,0,0,...,NaN,NaN,NaN,0,0,NaN,0,NaN,NaN,0.0


Structurally empty rows removed: 3988
Removed the entirely missing occupationId column.
Remaining rows: 1044597


### 2.2 Duplicate checks

**Note:** Check complete duplicates and repeated non-missing Job IDs. Do not delete similar titles: different postings may advertise the same role. Final Job ID uniqueness is checked again before export.

In [17]:
print("Complete duplicate rows:", clean.duplicated().sum())
print("Repeated non-missing Job IDs:",
      clean["metadata_jobPostId"].dropna().duplicated().sum())

Complete duplicate rows: 0
Repeated non-missing Job IDs: 0


### 2.3 Salary types and values

**Note:** Convert salary fields to numbers before inspecting their values. Preserve the source values and report any conversion failures. Do not fill missing salaries with an estimated wage.

In [18]:
for column in ["salary_minimum", "salary_maximum", "average_salary"]:
    clean[column + "_raw"] = clean[column]
    converted = pd.to_numeric(clean[column], errors="coerce")
    failed = clean[column].notna() & converted.isna()
    print(column, "conversion failures:", failed.sum())
    clean[column] = converted

display(clean[["salary_minimum", "salary_maximum"]].describe())

salary_minimum conversion failures: 0
salary_maximum conversion failures: 0
average_salary conversion failures: 0


,salary_minimum,salary_maximum
count,1.044597e+06,1.044597e+06
mean,3.829878e+03,5.745429e+03
std,3.169443e+03,5.027833e+04
min,1.000000e+00,1.000000e+00
25%,2.500000e+03,3.300000e+03
50%,3.000000e+03,4.500000e+03
75%,4.500000e+03,6.500000e+03
max,3.500000e+05,2.533000e+07


## Part 3: Data Transformation
### 3.1 Clean text and convert dates

**Note:** Remove surrounding and repeated spaces without changing job meanings. Convert dates for monthly analysis and report failed conversions.

In [19]:
clean["title_raw"] = clean["title"]
clean["position_level_raw"] = clean["positionLevels"]
clean["title_clean"] = clean["title"].str.strip().str.replace(r"\s+", " ", regex=True)
clean["position_level"] = clean["positionLevels"].str.strip().str.replace(r"\s+", " ", regex=True)

for column in ["metadata_expiryDate", "metadata_newPostingDate", "metadata_originalPostingDate"]:
    converted = pd.to_datetime(clean[column], errors="coerce")
    print(column, "conversion failures:", (clean[column].notna() & converted.isna()).sum())
    clean[column] = converted

clean["posting_month"] = clean["metadata_originalPostingDate"].dt.to_period("M").astype("string")
display(clean[["title_clean", "position_level", "posting_month"]].head())

metadata_expiryDate conversion failures: 0
metadata_newPostingDate conversion failures: 0
metadata_originalPostingDate conversion failures: 0


,title_clean,position_level,posting_month
0,Food Technologist - Clementi | Entry Level | U...,Executive,2023-03
1,"Software Engineer (Fab Support) (Java, CIM, Up...",Executive,2023-04
2,Senior Technician,Senior Executive,2023-04
3,"Senior .NET Developer (.NET Core, MVC, MVVC, S...",Senior Executive,2023-04
4,Sales / Admin Cordinator,Non-executive,2023-04


### 3.2 Extract categories and select the study population

**Note:** Categories are stored as JSON text. Extract the category names, then select source-labelled IT jobs with Monthly salaries. This defines the study population; other categories or salary periods are not data errors. Empty or unreadable category lists cannot establish IT membership.

In [20]:
def parse_categories(value):
    if pd.isna(value):
        return []
    try:
        items = json.loads(value)
    except (ValueError, TypeError):
        return []
    if not isinstance(items, list):
        return []
    return [item["category"] for item in items
            if isinstance(item, dict) and item.get("category") is not None]

clean["category_list"] = clean["categories"].apply(parse_categories)
clean["category_count"] = clean["category_list"].apply(len)
print("Rows without readable category names:", clean["category_count"].eq(0).sum())
is_it = clean["category_list"].apply(lambda names: "Information Technology" in names)
clean = clean.loc[is_it].copy()
print("IT rows:", len(clean))
display(clean["salary_type"].value_counts(dropna=False))
clean = clean.loc[clean["salary_type"].eq("Monthly")].copy()
print("Monthly IT rows:", len(clean))

Rows without readable category names: 0
IT rows: 140866


Monthly    140866
Name: salary_type, dtype: int64

Monthly IT rows: 140866


### 3.3 Classify IT job functions

**Note:** `position_level` describes seniority, while `it_job_category` describes the main job function inferred from the cleaned job title. The categories separate common IT functions so that salary comparisons are made between more comparable roles. Titles that do not match a clear function are retained as `Other IT` rather than being removed.

In [21]:
def classify_it_job(title):
    title = str(title).lower()

    if any(keyword in title for keyword in [
        "cyber", "cybersecurity", "information security", "security operations",
        "security engineer", "penetration", "soc analyst"
    ]):
        return "Cybersecurity"
    if any(keyword in title for keyword in [
        "data analyst", "data scientist", "data engineer", "business intelligence",
        "bi analyst", "analytics", "machine learning", "artificial intelligence"
    ]):
        return "Data & Analytics"
    if any(keyword in title for keyword in [
        "software", "developer", "programmer", "backend", "back-end", "frontend",
        "front-end", "full stack", "application developer", "web developer",
        "qa engineer", "test engineer", "automation engineer"
    ]):
        return "Software & Application Development"
    if any(keyword in title for keyword in [
        "cloud", "network", "infrastructure", "database", "devops", "systems engineer",
        "system engineer", "system administrator", "systems administrator", "network engineer",
        "database administrator", "platform engineer"
    ]):
        return "Infrastructure & Cloud"
    if any(keyword in title for keyword in [
        "helpdesk", "help desk", "desktop support", "it support", "technical support",
        "service desk", "application support", "support engineer"
    ]):
        return "IT Support & Systems"
    if any(keyword in title for keyword in [
        "it manager", "technology manager", "it project", "project manager",
        "business analyst", "systems analyst", "system analyst", "solution architect",
        "solutions architect", "it consultant", "technology consultant"
    ]):
        return "IT Management & Business Systems"
    return "Other IT"

clean["it_job_category"] = clean["title_clean"].apply(classify_it_job)
classification_check = (
    clean.groupby("it_job_category")
    .agg(
        posting_count=("metadata_jobPostId", "nunique"),
        percentage=("metadata_jobPostId", lambda values: round(values.nunique() / len(clean) * 100, 2)),
    )
    .sort_values("posting_count", ascending=False)
)
display(classification_check)
display(
    clean.loc[clean["it_job_category"].eq("Other IT"), ["title_clean", "position_level"]]
    .drop_duplicates()
    .head(20)
)


,posting_count,percentage
it_job_category,,
Other IT,63411,45.02
Software & Application Development,30117,21.38
Infrastructure & Cloud,17969,12.76
IT Management & Business Systems,12137,8.62
IT Support & Systems,7496,5.32
Cybersecurity,4998,3.55
Data & Analytics,4738,3.36


,title_clean,position_level
6,Urgent Hiring!!! Business Development Manager ...,Manager
10,"Electrical Engineer - Tuas | Generator exp|$4,700",Non-executive
39,Urgent / Service Delivery Manager - Japan MNC ...,Executive
44,Senior Consultant,Professional
75,UAT Tester,Professional
76,Urgent // Sales Manager (IT background / hunti...,Executive
77,IT Sales Manager / Global MNC / East,Senior Executive
100,Assistant Printing Engineer / Operator (produc...,Junior Executive
109,Marketing Executive,Executive
125,MNC Electrical Engineers / Project Engineer– M...,Senior Executive


### 3.4 Validate salary values

**Note:** Apply the salary quality rules after assigning IT job functions. A record is valid only when it has a Monthly salary type, both salary bounds are present, the minimum salary is greater than zero, and the maximum salary is at least the minimum salary. In `salary_valid`, `True` means that all checks pass and the record is retained; `False` means that at least one check fails and the record is excluded.

In [22]:
salary_valid = (
    clean["salary_type"].eq("Monthly")
    & clean[["salary_minimum", "salary_maximum"]].notna().all(axis=1)
    & clean["salary_minimum"].gt(0)
    & clean["salary_maximum"].ge(clean["salary_minimum"])
)

print("Invalid salary records:", (~salary_valid).sum())
clean = clean.loc[salary_valid].copy()
print("Retained postings after salary validation:", len(clean))

Invalid salary records: 0
Retained postings after salary validation: 140866


### 3.5 Create salary measures and summarise by IT job category

**Note:** Calculate the advertised salary midpoint and band width, then compare salary distributions by IT job function and position level. Medians are used as the main comparison because they are less affected by unusually high or low advertised ranges.

In [23]:
clean["salary_midpoint"] = (clean["salary_minimum"] + clean["salary_maximum"]) / 2
clean["salary_band_width"] = clean["salary_maximum"] - clean["salary_minimum"]

group_columns = ["it_job_category", "position_level"]
grouped_salary = clean.groupby(group_columns)["salary_midpoint"]
q1 = grouped_salary.transform(lambda values: values.quantile(0.25))
q3 = grouped_salary.transform(lambda values: values.quantile(0.75))
iqr = q3 - q1
group_count = grouped_salary.transform("count")
category_position_midpoint = grouped_salary.transform("median")

clean["salary_outlier_flag"] = (
    (group_count >= 5)
    & (
        (clean["salary_midpoint"] < q1 - 1.5 * iqr)
        | (clean["salary_midpoint"] > q3 + 1.5 * iqr)
    )
)
clean["category_position_midpoint"] = category_position_midpoint

outlier_data = (
    clean.loc[clean["salary_outlier_flag"]].copy()
    .sort_values(group_columns + ["salary_midpoint"], ascending=[True, True, False])
)
outlier_summary = (
    outlier_data.groupby(group_columns, dropna=False)
    .agg(
        outlier_count=("metadata_jobPostId", "nunique"),
        category_position_midpoint=("category_position_midpoint", "first"),
        highest_salary_midpoint=("salary_midpoint", "max"),
        lowest_salary_midpoint=("salary_midpoint", "min"),
    )
    .sort_values("outlier_count", ascending=False)
)

print("Potential salary outliers:", len(outlier_data))
display(outlier_summary.round(2))
display(
    outlier_data[
        group_columns + [
            "metadata_jobPostId", "title_clean", "salary_minimum", "salary_maximum",
            "salary_midpoint", "category_position_midpoint"
        ]
    ].head(20).round(2)
)

salary_by_category_level = (
    clean.groupby(group_columns, dropna=False)
    .agg(
        posting_count=("metadata_jobPostId", "nunique"),
        median_minimum_salary=("salary_minimum", "median"),
        median_maximum_salary=("salary_maximum", "median"),
        median_salary_midpoint=("salary_midpoint", "median"),
        q1_salary_midpoint=("salary_midpoint", lambda values: values.quantile(0.25)),
        q3_salary_midpoint=("salary_midpoint", lambda values: values.quantile(0.75)),
        median_salary_band_width=("salary_band_width", "median"),
        flagged_outliers=("salary_outlier_flag", "sum"),
    )
    .sort_values("median_salary_midpoint", ascending=False)
)

display(salary_by_category_level.round(2))


Potential salary outliers: 5138


outlier_count  \
it_job_category                    position_level                     
Software & Application Development Professional                 731   
Other IT                           Professional                 675   
                                   Executive                    488   
Infrastructure & Cloud             Professional                 380   
Other IT                           Manager                      276   
...                                                             ...   
Data & Analytics                   Junior Executive               3   
IT Support & Systems               Middle Management              2   
                                   Manager                        2   
Data & Analytics                   Fresh/entry level              1   
                                   Senior Management              1   

                                                      category_position_midpoint  \
it_job_category                    position_level                                  
Software & Application Development Professional                           8000.0   
Other IT                           Professional                           8250.0   
                                   Executive                              4250.0   
Infrastructure & Cloud             Professional                           7750.0   
Other IT                           Manager                                8100.0   
...                                                                          ...   
Data & Analytics                   Junior Executive                       4750.0   
IT Support & Systems               Middle Management                      7500.0   
                                   Manager                                6000.0   
Data & Analytics                   Fresh/entry level                      4250.0   
                                   Senior Management                     12500.0   

                                                      highest_salary_midpoint  \
it_job_category                    position_level                               
Software & Application Development Professional                      190000.0   
Other IT                           Professional                      225000.0   
                                   Executive                         215000.0   
Infrastructure & Cloud             Professional                      172500.0   
Other IT                           Manager                          1498693.5   
...                                                                       ...   
Data & Analytics                   Junior Executive                   18000.0   
IT Support & Systems               Middle Management                  31000.0   
                                   Manager                            17500.0   
Data & Analytics                   Fresh/entry level                  60000.0   
                                   Senior Management                 200000.5   

                                                      lowest_salary_midpoint  
it_job_category                    position_level                             
Software & Application Development Professional                          1.0  
Other IT                           Professional                      16900.0  
                                   Executive                         10175.0  
Infrastructure & Cloud             Professional                          8.5  
Other IT                           Manager                           16900.0  
...                                                                      ...  
Data & Analytics                   Junior Executive                   9500.0  
IT Support & Systems               Middle Management                 18000.0  
                                   Manager                           15000.0  
Data & Analytics                   Fresh/entry level                 60000.0  
                                   

,it_job_category,position_level,metadata_jobPostId,title_clean,salary_minimum,salary_maximum,salary_midpoint,category_position_midpoint
345863,Cybersecurity,Executive,MCF-2023-0644946,"enior Business Information Security Officer, G...",17000,25000,21000.0,5250.0
726438,Cybersecurity,Executive,MCF-2024-0124390,"Senior Business Information Security Officer, ...",17000,25000,21000.0,5250.0
876062,Cybersecurity,Executive,MCF-2024-0424525,"Manual Ethical Hacking Specialist, Global Info...",9500,19000,14250.0,5250.0
1031149,Cybersecurity,Executive,MCF-2024-0743448,"Manual Ethical Hacking Specialist, Global Info...",9500,19000,14250.0,5250.0
173656,Cybersecurity,Executive,MCF-2023-0473441,Senior Cyber Security Engineer,9000,18000,13500.0,5250.0
230103,Cybersecurity,Executive,MCF-2023-0531698,Senior Cyber Security Engineer,9000,18000,13500.0,5250.0
242488,Cybersecurity,Executive,MCF-2023-0543371,Network Security Engineer,10000,15000,12500.0,5250.0
12752,Cybersecurity,Executive,MCF-2023-0281819,Chief Information Security Officer (CISO) @ Ra...,12000,12000,12000.0,5250.0
13488,Cybersecurity,Executive,MCF-2023-0281832,Chief Information Security Officer (CISO) @ Ra...,12000,12000,12000.0,5250.0
527420,Cybersecurity,Executive,MCF-2023-0826358,"Cybersecurity Perimeter Response Team Analyst,...",8000,16000,12000.0,5250.0


,,posting_count,median_minimum_salary,median_maximum_salary,median_salary_midpoint,q1_salary_midpoint,q3_salary_midpoint,median_salary_band_width,flagged_outliers
it_job_category,position_level,,,,,,,,
Cybersecurity,Senior Management,143,12000.0,15000.0,14000.0,11375.0,16000.0,3200.0,6
Other IT,Senior Management,1891,10000.0,15000.0,12500.0,10000.0,17000.0,4000.0,31
Data & Analytics,Senior Management,65,10000.0,14980.0,12500.0,9100.0,18000.0,4500.0,1
Infrastructure & Cloud,Senior Management,272,9250.0,13000.0,11500.0,10000.0,13500.0,3000.0,19
IT Management & Business Systems,Senior Management,242,8500.0,12000.0,10000.0,8750.0,12500.0,3000.0,12
...,...,...,...,...,...,...,...,...,...
IT Support & Systems,Non-executive,514,2500.0,3500.0,3000.0,2750.0,3537.5,1000.0,77
IT Management & Business Systems,Fresh/entry level,272,2000.0,3500.0,2900.0,2750.0,3500.0,1500.0,82
Other IT,Fresh/entry level,3363,2200.0,3100.0,2750.0,1400.0,3600.0,700.0,85


## Part 4: Export and Check
### 4.1 Prepare the team outputs

**Note:** Keep the existing experience and date flags for team use. These flags do not remove records. The category table has one row per job–category pair, so count unique Job IDs when combining categories.

In [24]:
clean["experience_review_flag"] = clean["minimumYearsExperience"] >= 30
clean["from_may_2023_flag"] = clean["metadata_originalPostingDate"] >= "2023-05-01"

job_categories = (
    clean[["metadata_jobPostId", "category_list"]]
    .explode("category_list")
    .rename(columns={"category_list": "category"})
    .dropna(subset=["category"])
    .drop_duplicates()
    .reset_index(drop=True)
)

assert clean["metadata_jobPostId"].notna().all()
assert clean["metadata_jobPostId"].is_unique
assert clean["salary_minimum"].gt(0).all()
assert clean["salary_maximum"].ge(clean["salary_minimum"]).all()
assert clean["salary_type"].eq("Monthly").all()
assert clean["salary_midpoint"].notna().all()
assert clean["category_position_midpoint"].notna().all()
assert clean["it_job_category"].notna().all()
assert len(outlier_data) == clean["salary_outlier_flag"].sum()
print("Final validation passed.")


Final validation passed.


### 4.2 Save CSV files and read them back

**Note:** Use `index=False` to avoid exporting an extra index column. CSV does not preserve date types, so specify `parse_dates` when reading dates back.

In [26]:
clean.to_csv("jobs_clean.csv", index=False)
outlier_data.to_csv("salary_outliers.csv", index=False)
job_categories.to_csv("job_categories.csv", index=False)

saved = pd.read_csv("jobs_clean.csv", low_memory=False,
                    parse_dates=["metadata_originalPostingDate"])
saved_outliers = pd.read_csv("salary_outliers.csv", low_memory=False)
saved_categories = pd.read_csv("job_categories.csv")
assert saved.shape == clean.shape
assert saved["metadata_jobPostId"].is_unique
assert saved["metadata_jobPostId"].tolist() == clean["metadata_jobPostId"].tolist()
pd.testing.assert_frame_equal(
    saved[["salary_minimum", "salary_maximum", "salary_midpoint",
           "category_position_midpoint"]].reset_index(drop=True),
    clean[["salary_minimum", "salary_maximum", "salary_midpoint",
           "category_position_midpoint"]].reset_index(drop=True),
    check_dtype=False
)
assert len(saved_outliers) == len(outlier_data)
assert saved_outliers["category_position_midpoint"].notna().all()
pd.testing.assert_frame_equal(saved_categories, job_categories, check_dtype=False)
print("Raw postings:", len(raw))
print("Final monthly IT postings:", len(saved))
print("Salary outliers saved:", len(saved_outliers))
print("Saved and checked: jobs_clean.csv, salary_outliers.csv, job_categories.csv")


Raw postings: 1048585
Final monthly IT postings: 140866
Salary outliers saved: 5138
Saved and checked: jobs_clean.csv, salary_outliers.csv, job_categories.csv


## Conclusion

The exported dataset contains source-labelled IT postings with valid Monthly salary ranges and a new `it_job_category` field based on job-title functions. Salary comparisons are made across IT job functions and position levels using advertised salary midpoints, medians, and salary ranges. The IQR rule flags 5,138 potential salary outliers within `it_job_category × position_level` groups; these records are retained for review rather than automatically removed. The `category_position_midpoint` column is the median advertised salary midpoint for each category-position group. Salary figures describe advertised ranges, not actual employee pay.
